In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
import torch
print('torch', torch.__version__, 'cuda', torch.version.cuda)
print('gpu', torch.cuda.get_device_name(0), 'gb', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'trimesh', 'einops', 'pyyaml', 'opencv-python-headless', 'imageio', 'scikit-image', 'huggingface_hub', 'tinyobjloader'])
ver = torch.__version__.split('+')[0]
cu = 'cu' + torch.version.cuda.replace('.', '')
index = f'https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-{ver}_{cu}.html'
print('kaolin', index)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'kaolin', '-f', index])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-build-isolation', 'git+https://github.com/NVlabs/nvdiffrast.git'])


In [ ]:
root = Path('/kaggle/working/ReMu')
if not root.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/eth-ait/ReMu.git', str(root)])
sith = root / 'third_party' / 'sith'
data = sith / 'data'
data.mkdir(parents=True, exist_ok=True)
src = Path('/kaggle/input/dressme-remu-inputs')
shutil.copy(src / 'templates' / 'smplx_canonical.obj', data / 'smplx_canonical.obj')
shutil.copy(src / 'templates' / 'smplx_watertight.pkl', data / 'smplx_watertight.pkl')
test = Path('/kaggle/working/test')
if test.exists():
    shutil.rmtree(test)
for name in ('images', 'back_images', 'smplx'):
    shutil.copytree(src / name, test / name)
ckpt = root / 'checkpoints' / 'sith'
ckpt.mkdir(parents=True, exist_ok=True)
weights = ckpt / 'recon_model.pth'
if not weights.exists():
    subprocess.check_call(['curl', '-L', '--fail', '-o', str(weights), 'https://files.ait.ethz.ch/projects/SiTH/recon_model.pth'])
env = os.environ.copy()
last = 1
for grid in (512, 384, 256):
    out = Path(f'/kaggle/working/meshes_{grid}')
    if out.exists():
        shutil.rmtree(out)
    staged = Path(f'/kaggle/working/run_{grid}')
    if staged.exists():
        shutil.rmtree(staged)
    shutil.copytree(test, staged)
    print('reconstruct grid', grid)
    result = subprocess.run([
        sys.executable, 'reconstruct.py',
        '--test_folder', str(staged),
        '--config', str(sith / 'recon' / 'config.yaml'),
        '--resume', str(weights),
        '--grid_size', str(grid),
    ], cwd=str(sith), env=env)
    last = result.returncode
    meshes = staged / 'meshes'
    if result.returncode == 0 and meshes.exists() and any(meshes.glob('*.obj')):
        shutil.copytree(meshes, out)
        print('saved', out)
        break
    print('grid', grid, 'failed', result.returncode)
if last != 0:
    raise SystemExit(last)
